# 04 — Comparing finished runsThis notebook reads artifacts. Nothing is retrained, so what it shows is exactlywhat was recorded — including the runs where the holdout contradicted thedevelopment gate.Produce runs first:```bashuv run stock-movement run-all --config configs/reproduction/readme_aapl_2026_07.yamluv run stock-movement run-all --config configs/experiments/msft.yaml```

In [ ]:
import sys, json, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")from pathlib import Pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltruns = sorted(p for p in Path("../artifacts/runs").iterdir() if (p / "final_test_metrics.json").exists())print(f"{len(runs)} completed run(s):")for run in runs:    print(" ", run.name)

## Development gate versus final-test outcome

In [ ]:
rows = []for run in runs:    cfg = json.loads((run / "resolved_config.json").read_text())    decision = json.loads((run / "selection_decision.json").read_text())    metrics = json.loads((run / "final_test_metrics.json").read_text())    backtest = json.loads((run / "backtest_metrics.json").read_text())    model_key = next(k for k in metrics if not k.startswith("baseline_"))    baselines = {k: v for k, v in metrics.items() if k.startswith("baseline_")}    best_baseline = max(baselines.values(), key=lambda v: v["balanced_accuracy"])["balanced_accuracy"]    intraday = next((k for k in backtest if k.startswith("always_long")), None)    rows.append({        "run": cfg.get("run_name") or run.name,        "ticker": cfg["data"]["ticker"],        "edge_dev": decision["edge_detected"],        "test_balanced": metrics[model_key]["balanced_accuracy"],        "best_baseline": best_baseline,        "margin": metrics[model_key]["balanced_accuracy"] - best_baseline,        "model_return": backtest[model_key]["cumulative_return"],        "always_active": backtest[intraday]["cumulative_return"] if intraday else np.nan,        "buy_and_hold": backtest["buy_and_hold_close_to_close"]["cumulative_return"],        "exposure": backtest[model_key]["exposure"],    })summary = pd.DataFrame(rows).sort_values("margin")summary.round(4)

## Does the development gate predict the holdout?If the gate were informative, `edge_dev == True` would cluster on the positiveside of zero. It does not.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))colors = ["seagreen" if m > 0 else "indianred" for m in summary["margin"]]ax.bar(range(len(summary)), summary["margin"], color=colors)ax.axhline(0, color="black", lw=1.5)ax.set_xticks(range(len(summary)),              [f"{r.run}\n{'gate: yes' if r.edge_dev else 'gate: no'}" for r in summary.itertuples()],              fontsize=7, rotation=30, ha="right")ax.set_ylabel("final-test balanced accuracy\nminus best baseline")ax.set_title("Above zero = the model beat every baseline out of sample")ax.grid(alpha=0.3, axis="y")plt.tight_layout()

## Accuracy is not profitPlot the classification margin against the money. If accuracy translated intoreturn, these would trend together.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))ax.scatter(summary["margin"], summary["model_return"], s=90, zorder=3)for r in summary.itertuples():    ax.annotate(r.run, (r.margin, r.model_return), fontsize=7,                xytext=(5, 5), textcoords="offset points")ax.axhline(0, color="black", lw=1)ax.axvline(0, color="black", lw=1)ax.set_xlabel("final-test balanced accuracy minus best baseline")ax.set_ylabel("strategy cumulative return (net)")ax.set_title("Better direction calls did not mean better returns")ax.grid(alpha=0.3)plt.tight_layout()print(summary[["run", "margin", "model_return", "buy_and_hold"]].round(4).to_string(index=False))

## Every run loses to simply holding the asset

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))x = np.arange(len(summary))ax.bar(x - 0.2, summary["model_return"] * 100, width=0.4, label="model strategy")ax.bar(x + 0.2, summary["buy_and_hold"] * 100, width=0.4, label="buy and hold", color="black", alpha=0.7)ax.axhline(0, color="black", lw=1)ax.set_xticks(x, summary["run"], fontsize=7, rotation=30, ha="right")ax.set_ylabel("cumulative return (%)")ax.set_title("Model strategy vs simply holding, net of costs")ax.legend()ax.grid(alpha=0.3, axis="y")plt.tight_layout()

## Bootstrap intervals: does anything exclude zero?

In [ ]:
for run in runs:    name = json.loads((run / "resolved_config.json").read_text()).get("run_name") or run.name    boot = json.loads((run / "bootstrap_summary.json").read_text())    entry = boot["mean_daily_return"]    print(f"{name:32s} {entry['point']:+.6f} [{entry['ci_low']:+.6f}, {entry['ci_high']:+.6f}]"          f"  excludes zero: {entry['interval_excludes_zero']}")

## ConclusionRead any run's `model_card.md` for its recorded verdict. Across tickers, featuresets, model families, threshold policies and execution assumptions, noconfiguration demonstrates a stable out-of-sample edge, and every bootstrapinterval includes zero.That is a valid result, and reporting it is the point.